# MartiniSurf Colab: DNA + auto_martini M2


## 1) Install MartiniSurf

In [ ]:
#@title Install MartiniSurf (supports private repo via token)
REPO_URL = "https://github.com/jjimenezgar/MartiniSurf.git" #@param {type:"string"}
PRIVATE_REPO = True #@param {type:"boolean"}
INSTALL_VIEWER = True #@param {type:"boolean"}
ENSURE_MARTINIZE2 = True #@param {type:"boolean"}
ENSURE_DSSP_TOOL = False #@param {type:"boolean"}
ENSURE_PYTHON2_FOR_DNA = True #@param {type:"boolean"}

import getpass
import shlex
import shutil
import subprocess
from pathlib import Path


def run_cmd(cmd, check=True):
    if isinstance(cmd, str):
        res = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        shown = cmd
    else:
        res = subprocess.run(cmd, text=True, capture_output=True)
        shown = ' '.join(shlex.quote(x) for x in cmd)
    if check and res.returncode != 0:
        raise RuntimeError(f"Command failed ({shown})\nSTDOUT:\n{res.stdout[-4000:]}\nSTDERR:\n{res.stderr[-4000:]}")
    return res


def ensure_python2():
    if shutil.which('python2.7') or shutil.which('python2'):
        return

    mm = Path('/usr/local/bin/micromamba')
    if not mm.exists():
        run_cmd('curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /tmp bin/micromamba')
        run_cmd(['install', '-m', '755', '/tmp/bin/micromamba', '/usr/local/bin/micromamba'])

    env_prefix = Path('/content/py2env')
    if not (env_prefix / 'bin' / 'python').exists():
        run_cmd(['/usr/local/bin/micromamba', 'create', '-y', '-p', str(env_prefix), 'python=2.7'])

    wrapper = Path('/usr/local/bin/python2')
    wrapper.write_text('#!/usr/bin/env bash\n/content/py2env/bin/python "$@"\n')
    wrapper.chmod(0o755)

    py27 = Path('/usr/local/bin/python2.7')
    if not py27.exists():
        py27.symlink_to(wrapper)

run_cmd('rm -rf /content/MartiniSurf', check=False)

if PRIVATE_REPO:
    token = getpass.getpass('Paste your GitHub token (repo read access): ').strip()
    if not token:
        raise RuntimeError('Token is required for private repository installation.')
    if REPO_URL.startswith('https://'):
        clone_url = REPO_URL.replace('https://', f'https://{token}@', 1)
    else:
        raise RuntimeError('REPO_URL must use https://')
else:
    clone_url = REPO_URL

run_cmd(['git', 'clone', clone_url, '/content/MartiniSurf'])
run_cmd(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
run_cmd(['python', '-m', 'pip', 'install', '-q', '-e', '/content/MartiniSurf'])

if ENSURE_MARTINIZE2 and shutil.which('martinize2') is None:
    attempts = [
        ['python', '-m', 'pip', 'install', '-q', 'martinize2'],
        ['python', '-m', 'pip', 'install', '-q', 'vermouth'],
    ]
    for cmd in attempts:
        run_cmd(cmd, check=False)
        if shutil.which('martinize2') is not None:
            break

if ENSURE_DSSP_TOOL and shutil.which('mkdssp') is None:
    run_cmd(['apt-get', 'update', '-qq'], check=False)
    run_cmd(['apt-get', 'install', '-y', '-qq', 'dssp'], check=False)

if ENSURE_PYTHON2_FOR_DNA:
    ensure_python2()

if INSTALL_VIEWER:
    run_cmd(['python', '-m', 'pip', 'install', '-q', 'py3Dmol'])

print('Installed.')
print('martinisurf:', shutil.which('martinisurf') or 'NOT FOUND')
print('martinize2:', shutil.which('martinize2') or 'NOT FOUND')
print('python2:', shutil.which('python2') or 'NOT FOUND')
print('python2.7:', shutil.which('python2.7') or 'NOT FOUND')
print('mkdssp:', shutil.which('mkdssp') or 'NOT FOUND')


## 2) Optional linker generation (SMILES or uploaded file)

In [ ]:
#@title Optional linker generation (auto_martini M2)
RUN_LINKER_GENERATION = False #@param {type:"boolean"}
INPUT_MODE = "SMILES" #@param ["SMILES", "Upload file"]
SMILES = "CCO" #@param {type:"string"}
MOLECULE_NAME = "LNK" #@param {type:"string"}

import subprocess
from pathlib import Path

if RUN_LINKER_GENERATION:
    # Friendly upload mode kept. For robust M2 execution, we run from SMILES.
    if INPUT_MODE == 'Upload file':
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No file uploaded.')
        up_name = next(iter(uploaded.keys()))
        up_path = Path('/content') / up_name
        if up_path.suffix.lower() in {'.smi', '.txt'}:
            raw = up_path.read_text().strip().split()
            if raw:
                SMILES = raw[0]
            else:
                raise RuntimeError('Uploaded SMILES file is empty.')
        else:
            raise RuntimeError(
                'Upload mode currently supports .smi/.txt with a SMILES string. '
                'For reliable M2 execution use SMILES mode.'
            )

    if not SMILES.strip():
        raise RuntimeError('SMILES is empty.')

    # Install auto_martini directly from GitHub
    subprocess.run([
        'pip', 'install', '-q', 'git+https://github.com/tbereau/auto_martini'
    ], check=True)

    cmd = [
        'python',
        '-m',
        'auto_martini',
        '--mode', 'run',
        '--smi', SMILES,
        '--mol', MOLECULE_NAME,
        '--top', f'{MOLECULE_NAME}.itp',
        '--cg', f'{MOLECULE_NAME}.gro',
    ]

    print('Running:', ' '.join(cmd))
    run = subprocess.run(cmd, capture_output=True, text=True)

    if run.returncode != 0:
        print(run.stderr[-4000:])
        raise RuntimeError('auto_martini M2 failed.')

    print(f'Generated: {MOLECULE_NAME}.itp and {MOLECULE_NAME}.gro')
    subprocess.run(['ls', '-lh', f'{MOLECULE_NAME}.*'])
else:
    print('Skipped.')


## 3) Build system

In [ ]:
#@title Build DNA system (ordered form, unlimited groups)
# ==============================
# 1) Input
# ==============================
PDB_INPUT_MODE = "PDB ID or filename" #@param ["PDB ID or filename", "Upload PDB file"]
PDB_INPUT = "4C64.pdb" #@param {type:"string"}
DNATYPE = "ds-stiff" #@param ["ds-stiff", "ds-flex", "ss"]
MERGE_GROUPS = "" #@param {type:"string"}
OUTDIR = "Simulation_Files" #@param {type:"string"}

# ==============================
# 2) Surface
# ==============================
USE_EXISTING_SURFACE = True #@param {type:"boolean"}
SURFACE_SOURCE = "Upload surface.gro now" #@param ["Upload surface.gro now", "Use existing surface.gro in session"]
SURFACE_FILE = "surface.gro" #@param {type:"string"}
# Choose lattice preset for generated surfaces:
# - 4-1: denser lattice (common dx ~0.27 nm)
# - 2-1: more open lattice (common dx ~0.47 nm)
SURFACE_TOPOLOGY = "4-1" #@param ["4-1", "2-1"]
SURFACE_DX_NM = 0.27 #@param {type:"number"}
LX = 5.0 #@param {type:"number"}
LY = 5.0 #@param {type:"number"}
SURFACE_BEAD = "C1" #@param {type:"string"}

# ==============================
# 3) Orientation mode
# ==============================
# Orientation mode tip:
# - Anchor mode: direct biomolecule-to-surface anchoring
# - Linker mode: uses linker.gro/.itp and linker groups
ORIENTATION_MODE = "Anchor mode" #@param ["Anchor mode", "Linker mode"]
DIST = 10.0 #@param {type:"number"}

# Unlimited anchor groups: separate groups with ; or newline
# Format per group: GROUP_ID RESID [RESID ...]
ANCHOR_GROUPS = "1 1; 2 24" #@param {type:"string"}

# Unlimited linker groups: separate groups with ; or newline
# Format per group: GROUP_ID RESID [RESID ...]
LINKER_GROUPS = "1 1" #@param {type:"string"}

# ==============================
# 4) Linker files (only Linker mode)
# ==============================
LINKER_SOURCE = "Upload linker files now" #@param ["Upload linker files now", "Use existing linker.gro/.itp in session"]
LINKER_GRO = "linker.gro" #@param {type:"string"}
INVERT_LINKER = False #@param {type:"boolean"}
SURFACE_LINKERS = 0 #@param {type:"integer"}

# ==============================
# 5) Run
# ==============================
DRY_RUN = False #@param {type:"boolean"}

import shlex
import shutil
import subprocess
from pathlib import Path


def parse_group_lines(raw_text, field_name):
    groups = []
    for ln in (raw_text or '').replace(';', '\n').splitlines():
        line = ln.strip()
        if not line:
            continue
        parts = line.replace(',', ' ').split()
        if len(parts) < 2:
            raise ValueError(f'{field_name}: invalid line "{line}". Use: GROUP RESID [RESID ...]')
        groups.append(parts)
    return groups


def parse_merge_groups(raw_text):
    vals = []
    for token in (raw_text or '').replace(';', '\n').splitlines():
        item = token.strip()
        if item:
            vals.append(item)
    return vals

if shutil.which('martinisurf') is None:
    raise RuntimeError('martinisurf is not installed. Run the Install cell first.')

if PDB_INPUT_MODE == 'Upload PDB file':
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No PDB uploaded')
    pdb_input = '/content/' + next(iter(uploaded.keys()))
else:
    pdb_input = PDB_INPUT.strip()

if USE_EXISTING_SURFACE:
    print('Using existing surface file from session/upload.')
    if SURFACE_SOURCE == 'Upload surface.gro now':
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No surface file uploaded')
        surface_path = '/content/' + next(iter(uploaded.keys()))
    else:
        surface_path = '/content/' + SURFACE_FILE
        if not Path(surface_path).exists():
            raise FileNotFoundError(f'Missing file: {surface_path}')

cmd = ['martinisurf', '--dna', '--pdb', pdb_input, '--dnatype', DNATYPE, '--outdir', OUTDIR]

for mg in parse_merge_groups(MERGE_GROUPS):
    cmd += ['--merge', mg]

if USE_EXISTING_SURFACE:
    print('Using existing surface file from session/upload.')
    cmd += ['--surface', surface_path]
else:
    print(f'Generated surface preset: {SURFACE_TOPOLOGY} (dx={SURFACE_DX_NM} nm)')
    dx_nm = float(SURFACE_DX_NM)
    if dx_nm <= 0:
        raise ValueError('SURFACE_DX_NM must be > 0.')
    cmd += ['--lx', str(LX), '--ly', str(LY), '--dx', str(dx_nm), '--surface-bead', SURFACE_BEAD]

if ORIENTATION_MODE == 'Linker mode':
    if LINKER_SOURCE == 'Upload linker files now':
        from google.colab import files
        print('Upload linker.gro and linker.itp (matching basename preferred)')
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No linker files uploaded')
        names = list(uploaded.keys())
        gros = [n for n in names if n.lower().endswith('.gro')]
        itps = [n for n in names if n.lower().endswith('.itp')]
        if not gros:
            raise RuntimeError('Please upload one linker .gro file')
        if not itps:
            raise RuntimeError('Please upload one linker .itp file')
        linker_gro = Path('/content') / gros[0]
        linker_itp = Path('/content') / itps[0]
        target_itp = linker_gro.with_suffix('.itp')
        if linker_itp != target_itp:
            shutil.copyfile(linker_itp, target_itp)
        linker_path = str(linker_gro)
    else:
        linker_path = '/content/' + LINKER_GRO
        if not Path(linker_path).exists():
            raise FileNotFoundError(f'Missing linker file: {linker_path}')
        expected_itp = Path(linker_path).with_suffix('.itp')
        if not expected_itp.exists():
            raise FileNotFoundError(f'Missing linker topology: {expected_itp}')

    cmd += ['--linker', linker_path]
    if INVERT_LINKER:
        cmd += ['--invert-linker']
    if int(SURFACE_LINKERS) > 0:
        cmd += ['--surface-linkers', str(int(SURFACE_LINKERS))]

    groups = parse_group_lines(LINKER_GROUPS, 'LINKER_GROUPS')
    if not groups:
        raise ValueError('LINKER_GROUPS is empty. Add at least one group line.')
    for g in groups:
        cmd += ['--linker-group'] + g
else:
    groups = parse_group_lines(ANCHOR_GROUPS, 'ANCHOR_GROUPS')
    if not groups:
        raise ValueError('ANCHOR_GROUPS is empty. Add at least one group line.')
    for g in groups:
        cmd += ['--anchor'] + g
    cmd += ['--dist', str(DIST)]

print('Running:\n' + ' '.join(shlex.quote(x) for x in cmd))
if DRY_RUN:
    print('DRY_RUN=True, command not executed.')
else:
    res = subprocess.run(cmd, text=True, capture_output=True)
    print(res.stdout[-12000:])
    if res.returncode != 0:
        print(res.stderr[-12000:])
        combined = ((res.stdout or '') + '\n' + (res.stderr or '')).lower()
        msg = 'MartiniSurf failed. '
        if 'python2' in combined and ('not found' in combined or 'requires' in combined):
            msg += 'Likely cause: Python 2.7 is missing for DNA martinize-dna.py.'
        elif 'failed to download rcsb' in combined:
            msg += 'Likely cause: network issue or invalid PDB ID.'
        raise RuntimeError(msg)

    print('Build completed')


## 4) Download

In [ ]:
#@title Download results as ZIP
OUTPUT_FOLDER = "Simulation_Files" #@param {type:"string"}
ZIP_NAME = "Simulation_Files" #@param {type:"string"}

import shutil
from pathlib import Path
from google.colab import files

folder = Path('/content') / OUTPUT_FOLDER
if not folder.exists():
    raise FileNotFoundError(f'Folder not found: {folder}')

shutil.make_archive(ZIP_NAME, 'zip', str(folder))
files.download(ZIP_NAME + '.zip')


## 5) Visualize final structure

In [ ]:
#@title Viewer (generated structure)
SYSTEM_GRO_FILE = "Simulation_Files/2_system/immobilized_system.gro" #@param {type:"string"}
STYLE = "Spheres" #@param ["Spheres", "Sticks"]

import py3Dmol
from pathlib import Path

path = Path('/content') / SYSTEM_GRO_FILE
if not path.exists():
    raise FileNotFoundError(f'File not found: {path}')

model = path.read_text()
view = py3Dmol.view(width='100%', height=800)
view.addModel(model, 'gro')
if STYLE == 'Sticks':
    view.setStyle({'stick': {}})
else:
    view.setStyle({'sphere': {'radius': 1.4}})
view.zoomTo()
view.show()


## Acknowledgements, Licenses, and Citations
This notebook uses external tools and libraries with their own licenses and citation requirements.

### Please cite
- **martinize / martinize-dna workflow**: de Jong DH, Singh G, Bennett WFD, et al. *Improved Parameters for the Martini Coarse-Grained Protein Force Field*. J. Chem. Theory Comput. DOI: `10.1021/ct300646g`.
- **auto_martini (M2)**: Bereau T, Kremer K. *Automated parametrization of the coarse-grained Martini force field for small organic molecules*. Journal of Chemical Theory and Computation, 2015, 11(6):2783-2791.

### Licenses
- **MartiniSurf**: this repository license.
- **martinize-dna / Martini2 DNA tooling**: see upstream project/license notes.
- **auto_martini**: see upstream project license.
- Python packages used here (depending on workflow): `numpy`, `scipy`, `MDAnalysis`, `mdtraj`, `py3Dmol`.
